In [4]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check if 'print' has been overwritten as a string and delete it to restore the built-in.
# This error usually occurs when a variable named 'print' is assigned a non-callable value (like a string).
if 'print' in globals() and isinstance(globals()['print'], str):
    del print

print('Libraries ready')

Libraries ready


In [5]:
from groq import Groq
API_KEY="XXXXXX"
client=Groq(api_key=API_KEY)
MODEL='llama-3.1-8b-instant'
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [6]:
def ask_llm(
    user_message,
    system_message="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=1500 # user(trainer) defined words namma kudukarathu thaan
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content
test_response = ask_llm(
    "What is Catharanthus roseus? Answer in exactly 2 sentences."
)

print("=== LLM Response ===")
print(test_response)

=== LLM Response ===
Catharanthus roseus, also known as Madagascar periwinkle or old maid, is a flowering plant native to Madagascar, widely cultivated for its attractive flowers and used in traditional medicine for various purposes. It is a popular ornamental plant with vibrant purple, pink, red, or white flowers and is also the source of several medically useful compounds, including vincristine and vinblastine used in cancer treatment.


In [7]:
response_etl=ask_llm(# user message
    "In 3 bullet points,explain the chemical components in Catharanthus roseus"
    "How Catharanthus roseus used in medical field",
    system_message="You are a Biology Student"#system message ,system's role to get relevant answer
    "Be consise and stay desire of learning new"
)
print("=== LLM Response ===")
print("Catharanthus roseus")
print(response_etl)
print()
print('Token Explaination')
print('Each word is roughly 1-2 tokens')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')


=== LLM Response ===
Catharanthus roseus
Here are the chemical components and medical uses of Catharanthus roseus:

**Chemical Components:**

* Alkaloids: Catharanthine, Vinblastine, Vincristine, and Vinorelbine are the main alkaloids found in Catharanthus roseus. These alkaloids have antitumor properties and are used in cancer treatment.
* Glycosides: The plant also contains glycosides like ajugol and ajugoside, which have antimicrobial and antifungal properties.
* Flavonoids: Quercetin and kaempferol are the main flavonoids found in Catharanthus roseus, which have antioxidant and anti-inflammatory properties.

**Medical Uses:**

* **Cancer Treatment**: The alkaloids vinblastine, vincristine, and vinorelbine are used to treat various types of cancer, including leukemia, lymphoma, and solid tumors.
* **Antibiotic and Antifungal Agent**: The glycosides and flavonoids in Catharanthus roseus have been shown to have antimicrobial and antifungal properties, making them useful in treating in

In [8]:
response_etl=ask_llm(
    "What is Data Science?"
    "You are a Biology Student",
    system_message="You are a Biology Student"
    "Be consise and stay desire of learning new"
)
print("=== LLM Response ===")
print("Catharanthus roseus")
print(response_etl)
print()
print('Token Explaination')
print('Each word is roughly 1-2 tokens')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')


=== LLM Response ===
Catharanthus roseus
As a biology student, I've recently been introduced to the concept of Data Science. From what I understand, Data Science is an interdisciplinary field that combines statistics, computer science, and domain-specific knowledge to extract insights and knowledge from large datasets.

In biology, data science is used to analyze and interpret complex biological data, such as genomic sequences, gene expression patterns, and protein structures. It helps us identify patterns, trends, and correlations that can lead to new discoveries and a deeper understanding of biological systems.

Some of the key techniques used in data science include:

1. Machine learning: algorithms that enable computers to learn from data and make predictions or classifications.
2. Statistical analysis: methods for analyzing and interpreting data to identify patterns and trends.
3. Data visualization: techniques for presenting complex data in a clear and intuitive way.
4. Programmi

In [9]:
#Cleaning the data
zero_shot_response=ask_llm(
    "Extract the city name from this address:"
    "456 Brigade Road,Bangalore 560025,Karnataka,India"
)
print("Zero Shot LLM response:")
print(zero_shot_response)
print()
ambiguous_response=ask_llm("Clean this data:ramesh kumar,45000,mumbai")
print(ambiguous_response)
print()
print('Problem:Output format is unpredictable and not machine-parseable!!')

Zero Shot LLM response:
The city name is: Bangalore.

The data appears to be in a simple CSV (Comma Separated Values) format. Based on this, I'll break down the data and provide a cleaned version.

Original Data:
ramesh kumar,45000,mumbai

Cleaned Data:

- Name: Ramesh Kumar
- Salary: 45,000
- Location: Mumbai

Cleaned Data in a more structured format:

| Name         | Salary    | Location    |
|--------------|-----------|-------------|
| Ramesh Kumar | 45,000    | Mumbai      |

Note: I've assumed that the 'kumar' is part of the name and formatted it accordingly. If 'kumar' is a surname and 'ramesh' is the given name, the format would be different. 

However, based on general naming conventions in India, 'ramesh kumar' is likely to be the full name, with 'ramesh' as the given name and 'kumar' as the surname.

Problem:Output format is unpredictable and not machine-parseable!!


In [10]:
few_shot_prompt="""
Convert employee text to JSON.Here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"Ramesh Kumar","salary":45000,"city":"MUMBAI"}
Input:HARSHINI,80000,COIMBATORE
Output:{"name":"Harshini","salary":80000,"city":"Coimbatore"}
Input:Bala Kumar,80000,COIMBATORE
Output:
"""
few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print("Few Shot LLM response:")
print(few_shot_response)
print()
try:
  parsed=json.loads(few_shot_response.strip())
  print("Successfully parsed JSON!!")
  print(f"Name:{parsed['name']}")
  print(f"Salary:{parsed['salary']}")
  print(f"City:{parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed --> model added extra text")
  print("Solution:add explicit instructions in the system prompt")

Few Shot LLM response:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def employee_to_json(employee_text):
    """
    Convert employee text to JSON.

    Args:
        employee_text (str): Employee information in the format "name,salary,city".

    Returns:
        dict: Employee information in JSON format.
    """
    # Split the employee text into individual fields
    fields = employee_text.split(',')

    # Capitalize the first letter of each field and convert to title case
    name = fields[0].title()
    salary = fields[1]
    city = fields[2].upper()

    # Create a dictionary with the employee information
    employee_info = {
        "name": name,
        "salary": int(salary),
        "city": city
    }

    # Convert the dictionary to JSON
    employee_json = json.dumps(employee_info)

    return employee_json

# Test the function
print(employee_to_json("RAMESH KUMAR,45000,mumbai"))
print(employee_to_json("HARSHINI,80000,COIMBAT

In [11]:
## few shot prompt
#boy girl determination
few_shot_prompt="""
Tells the gender of a person by name,

Input :Harshini
Output:Female

Input:Bala Kumar
Output:Male

Input:Kanishka
Output:Female

Input:Sandhiya
Output:
State:Tamil Nadu

who is the Sandhiya?
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.8)
print("==== Few Shot Result====")
print(few_shot_response,'\n')

try:
  parsed = json.loads(few_shot_response.strip())
  print("Successfully parsed as JSON")
  print(f"Name: {parsed['name']}, Salary: {parsed['salary']}, City: {parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed - model added extra text")
  print("Solution: add explicit instructions in the system prompt")

==== Few Shot Result====
To determine the gender of a person by name, I'll try to use a combination of traditional Indian naming conventions and machine learning models. However, please note that this is not an exact science and may not always be accurate.

Based on your input:

Input :Harshini
Output: Female

Input:Bala Kumar
Output: Male

Input:Kanishka
Output: Female

Input:Sandhiya
Output: Female (not State:Tamil Nadu)

In Indian naming conventions, 'Sandhiya' is a female given name, commonly associated with the Tamil language and culture. It is a popular name in the Tamil Nadu region, but it is not exclusive to that region. Sandhiya is often used to refer to a woman who is born during the Sandhi or the in-between period of two days, which is considered an auspicious time in Hindu astrology. 

Parsing failed - model added extra text
Solution: add explicit instructions in the system prompt


In [12]:
same_question="Review this Python code and identify any issues:\n" \
              "df['revenue']=df['qty'] =df['price]\n" \
              "result=df.groupby(qty,price)"
generic_response=ask_llm(same_question,temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()
role_response=ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production "
    "experience.Review code critically for production readiness, "
    "data type issues,and potential failures at scale.",
temperature=0.2
)
print('With Role Promting (Senior Data Engineer):')
print(role_response[:400], '...')
print()
print('Notice: role prompting produces more technical, actionable feedback')

Without Role Prompting:
There are several issues with the provided Python code:

1. **Assignment Operator**: The line `df['revenue']=df['qty'] =df['price']` is using a single equals sign (`=`) which is an assignment operator. It's trying to assign the value of `df['price']` to both `df['revenue']` and `df['qty']`. However, ...

With Role Promting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a snippet from a larger data analysis or data engineering project. However, there are several issues that need to be addressed to make it production-ready:

```python
# Issue 1: Incorrect assignment
df['revenue'] = df['qty'] = df['price']

# This line is attempting to assign the value of 'price' to both 'revenue' and 'qty' columns.
# It's lik ...

Notice: role prompting produces more technical, actionable feedback


In [13]:
#temperature experiment
prompt="Give me one creative name for a data analytics startup."
print(' ===Temperature Experiment=== ')
for temp in [0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature{temp}:{response.strip()}')
  time.sleep(1)

print()
print("Observations:")
print('Temperature=0.0 -->same or very similar answer every run(determinstic)')
print('Temperature=0.5 -->some variation')
print('Temperature=1.0 -->more creative/varied,sometimes surprising')
print()
print('Rule for data engineering tasks:use temperature=0.0 or 0.1')
print('You need CONSISTENT,PARSABLE output -->not creative variation')

 ===Temperature Experiment=== 
Temperature0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature0.5:Here's a creative name for a data analytics startup: 

"InsightForge" 

This name suggests the idea of shaping and molding insights from data into valuable information, which aligns perfectly with the mission of a data analytics startup.
Temperature1.0:Here's a creative name for a data analytics startup: 

"Apexion Insights"

This name combines "Apex" (meaning the highest or most superior point) with a suffix that suggests expert analysis or deeper understanding, which is fitting for a data analytics company.

Observations:
Temperature=0.0 -->same or very similar answer every run(determinstic)
Temp

In [14]:
messy_invoices=[
    "INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45000 Laptop purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for office Cleaning services",
    "#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024|Python training",
    "SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20",
    "Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95000|Server hardware"
]
print('Messy Invoices to process:')
for i,inv in enumerate(messy_invoices,1):
  print(f'{i}.{inv}')
print(f'\n Total :{len(messy_invoices)} invoices')

Messy Invoices to process:
1.INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45000 Laptop purchase
2.Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for office Cleaning services
3.#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024|Python training
4.SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20
5.Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95000|Server hardware

 Total :5 invoices


In [15]:
EXTRACTION_SYSTEM_PROMPT = """
You are an expert invoice extraction system.
Extract invoice number, vendor name, date, amount, and description.
Return only valid JSON.
"""
print("Extraction prompt engineered successfully!")
print("The invoice extraction system prompt is ready and optimized for structured data extraction.\n")
print(f"System prompt length: {len(EXTRACTION_SYSTEM_PROMPT)} characters")
print(f"~{len(EXTRACTION_SYSTEM_PROMPT.split())} words, ~{int(len(EXTRACTION_SYSTEM_PROMPT.split()) * 1.3)} tokens")

Extraction prompt engineered successfully!
The invoice extraction system prompt is ready and optimized for structured data extraction.

System prompt length: 138 characters
~20 words, ~26 tokens


In [16]:
def extract_invoice_data(invoice_text,system_prompt,client,model):
    """
    Extract structured JSON from a single messy invoice string.
    Returns a Python dict or None on failure.
    """
    try:
        response=client.chat.completions.create(
            model=model,
            messages=[
                {"role":"system","content":system_prompt},
                {"role":"user","content":f"Extract from: {invoice_text}"}
            ],
            temperature=0.0,
            max_tokens=400
        )
        raw = response.choices[0].message.content.strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            pass
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        print(f"WARNING: Could not parse JSON from response: {raw[:80]}...")
        return None
    except Exception as e:
        print(f"Error calling API: {e}")
        return None
print("Processing invoices with LLM...")
extracted_records = []
for i, invoice in enumerate(messy_invoices, 1):
    print(f"\n[{i}/{len(messy_invoices)}]Input:{invoice[:70]}...")
    result=extract_invoice_data(
        invoice,EXTRACTION_SYSTEM_PROMPT,client,
        MODEL
    )
    if result:
        extracted_records.append(result)
        print(
            f"Extracted Vendor={result.get('vendor_name')}, "
            f"Amount={result.get('amount')}, "
            f"Date={result.get('invoice_date')}"
        )
    else:
        print("--> Failed, adding placeholder")
        extracted_records.append({
            "invoice_id":None,"vendor_name":"EXTRACTION_FAILED","amount":None,"currency":"INR","invoice_date":None,"category":"Other","description":invoice[:50]
        })
    time.sleep(0.5)
print(f"\nProcessed:{len(extracted_records)}/{len(messy_invoices)} invoices")

Processing invoices with LLM...

[1/5]Input:INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45000 Laptop purcha...
Extracted Vendor=TECHWORLD SOLUTIONS, Amount=45000, Date=None

[2/5]Input:Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for office Clea...
Extracted Vendor=PRIYA ENTERPRISES, Amount=12500, Date=None

[3/5]Input:#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024|Python t...
Extracted Vendor=arjun nair consultancy, Amount=8000, Date=None

[4/5]Input:SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01...
Extracted Vendor=SURESH RAO HARDWARE STORE, Amount=25000, Date=None

[5/5]Input:Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95...
Extracted Vendor=Ananya Tech Solutions, Amount=95000, Date=None

Processed:5/5 invoices


In [17]:
invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(
    invoices_df['amount'],
    errors='coerce'
)
invoices_df['invoice_date'] = pd.to_datetime(
    invoices_df['date'],
    errors='coerce'
)
invoices_df = invoices_df.drop(columns=['date'])
print("SMART DATA CLEANER OUTPUT")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}")
print()
print(invoices_df.to_string(index=False))

SMART DATA CLEANER OUTPUT
Rows: 5 | Columns: 5

                invoice_number               vendor_name  amount                    description invoice_date
                 INV-2024-0891       TECHWORLD SOLUTIONS   45000                Laptop purchase   2024-01-15
Invoice from PRIYA ENTERPRISES         PRIYA ENTERPRISES   12500       office Cleaning services          NaT
                 #INV-2024-103    arjun nair consultancy    8000                Python training          NaT
                          None SURESH RAO HARDWARE STORE   25000 Keyboard and Mouse accessories          NaT
                       INV-897     Ananya Tech Solutions   95000                Server hardware          NaT
